## 데이터 추가 확보

### 1. Google drive 연결

In [9]:
from google.colab import drive
drive.mount("/content/gdrive")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


### 2. 라이브러리 불러오기

In [10]:
import json
import zipfile
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

### 3. 데이터 경로 설정


추가 확보한 AI Hub 라벨링 ZIP 파일은 개인 Google Drive에서 관리하고,
기존 프로젝트의 56개 클래스 정보는 팀 프로젝트 Annotation에서 불러옵니다.

In [11]:
# 개인 Google Drive
MY_DRIVE = Path("/content/gdrive/MyDrive")

# 추가 AI Hub TL ZIP 저장 폴더
ADDITIONAL_DATA_DIR = (
    MY_DRIVE / "pill_additional_data"
)

# 기존 프로젝트 경로
PROJECT_DIR = (
    MY_DRIVE / "코드잇_파트2_3팀_프로젝트"
)

TRAIN_ANNOTATION_DIR = (
    PROJECT_DIR / "project1-data" / "train_annotations"
)

print("추가 데이터 경로:", ADDITIONAL_DATA_DIR)
print("기존 Annotation 경로:", TRAIN_ANNOTATION_DIR)

print("추가 데이터 폴더 존재:", ADDITIONAL_DATA_DIR.exists())
print("기존 Annotation 폴더 존재:", TRAIN_ANNOTATION_DIR.exists())

추가 데이터 경로: /content/gdrive/MyDrive/pill_additional_data
기존 Annotation 경로: /content/gdrive/MyDrive/코드잇_파트2_3팀_프로젝트/project1-data/train_annotations
추가 데이터 폴더 존재: True
기존 Annotation 폴더 존재: True


### 4. 추가 데이터 ZIP 파일 확인

AI Hub에서 다운로드한 TL 라벨링 데이터를 확인합니다.

사용이 금지된 `TL_2_조합.zip`은 분석 대상에서 제외합니다.

In [25]:
zip_files = sorted(
    [
        p for p in ADDITIONAL_DATA_DIR.glob("*.zip")
        if p.name != "TL_2_조합.zip"
    ]
)

print("확인할 ZIP 수:", len(zip_files))

for zip_path in zip_files:
    print(zip_path.name)

확인할 ZIP 수: 81
TL_10_단일.zip
TL_11_단일.zip
TL_12_단일.zip
TL_13_단일.zip
TL_14_단일.zip
TL_15_단일.zip
TL_16_단일.zip
TL_17_단일.zip
TL_18_단일.zip
TL_19_단일.zip
TL_1_단일.zip
TL_20_단일.zip
TL_21_단일.zip
TL_22_단일.zip
TL_23_단일.zip
TL_24_단일.zip
TL_25_단일.zip
TL_26_단일.zip
TL_27_단일.zip
TL_28_단일.zip
TL_29_단일.zip
TL_2_단일.zip
TL_30_단일.zip
TL_31_단일.zip
TL_32_단일.zip
TL_33_단일.zip
TL_34_단일.zip
TL_35_단일.zip
TL_36_단일.zip
TL_37_단일.zip
TL_38_단일.zip
TL_39_단일.zip
TL_3_단일.zip
TL_40_단일.zip
TL_41_단일.zip
TL_42_단일.zip
TL_43_단일.zip
TL_44_단일.zip
TL_45_단일.zip
TL_46_단일.zip
TL_47_단일.zip
TL_48_단일.zip
TL_49_단일.zip
TL_4_단일.zip
TL_50_단일.zip
TL_51_단일.zip
TL_52_단일.zip
TL_53_단일.zip
TL_54_단일.zip
TL_55_단일.zip
TL_56_단일.zip
TL_57_단일.zip
TL_58_단일.zip
TL_59_단일.zip
TL_5_단일.zip
TL_60_단일.zip
TL_61_단일.zip
TL_62_단일.zip
TL_63

### 5. 추가 라벨 데이터 구조 확인


대규모 데이터를 전체 탐색하기 전에 ZIP 내부 JSON 파일 하나를 샘플링하여
AI Hub 원본 Annotation 구조와 약품 식별 필드를 확인합니다.

In [13]:
sample_zip = None
sample_json_name = None

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        for name in z.namelist():
            if name.lower().endswith(".json"):
                sample_zip = zip_path
                sample_json_name = name
                break

    if sample_json_name is not None:
        break

print("샘플 ZIP:", sample_zip.name if sample_zip else None)
print("샘플 JSON:", sample_json_name)

샘플 ZIP: TL_1_조합.zip
샘플 JSON: K-000250-000573-002483-006192_json/K-000250/K-000250-000573-002483-006192_0_2_0_2_70_000_200.json


In [14]:
with zipfile.ZipFile(sample_zip, "r") as z:
    with z.open(sample_json_name) as f:
        sample_data = json.load(f)

print("최상위 Key:")
print(sample_data.keys())

최상위 Key:
dict_keys(['images', 'type', 'annotations', 'categories'])


In [15]:
if "images" in sample_data and sample_data["images"]:
    print(sample_data["images"][0])

{'file_name': 'K-000250-000573-002483-006192_0_2_0_2_70_000_200.png', 'width': 976, 'height': 1280, 'imgfile': 'K-000250-000573-002483-006192_0_2_0_2_70_000_200.png', 'drug_N': 'K-000250', 'drug_S': '정상알약', 'back_color': '연회색 배경', 'drug_dir': '앞면', 'light_color': '주백색', 'camera_la': 70, 'camera_lo': 0, 'size': 200, 'dl_idx': '249', 'dl_mapping_code': 'K-000250', 'dl_name': '마그밀정(수산화마그네슘)', 'dl_name_en': 'Magmil Tab. 500mg', 'img_key': 'http://connectdi.com/design/img/drug/1NgBQQRiuc_.jpg', 'dl_material': '수산화마그네슘', 'dl_material_en': 'Magnesium Hydroxide', 'dl_custom_shape': '정제, 저작정', 'dl_company': '삼남제약(주)', 'dl_company_en': 'Samnam', 'di_company_mf': '', 'di_company_mf_en': '', 'item_seq': 197400246, 'di_item_permit_date': '19740902', 'di_class_no': '[02340]제산제', 'di_etc_otc_code': '일반의약품', 'di_edi_code': '653700240,A11800371', 'chart': '흰색의 원형정제', 'drug_shape': '원형', 'thick': 4.5, 'leng_long': 11.1, 'leng_short': 11.1, 'print_front': '마크', 'print_back': '마크', 'color_class1': '하양', '

### 6. 기존 Annotation 경로 및 파일 확인

기존 프로젝트의 Annotation 경로가 올바르게 연결되었는지 확인하고,
하위 폴더를 포함하여 JSON 파일을 탐색합니다.

In [17]:
print("Annotation 경로:", TRAIN_ANNOTATION_DIR)
print("폴더 존재 여부:", TRAIN_ANNOTATION_DIR.exists())

base_json_files = list(TRAIN_ANNOTATION_DIR.rglob("*.json"))

print("JSON 파일 수:", len(base_json_files))

for path in base_json_files[:5]:
    print(path)

Annotation 경로: /content/gdrive/MyDrive/코드잇_파트2_3팀_프로젝트/project1-data/train_annotations
폴더 존재 여부: True
JSON 파일 수: 763
/content/gdrive/MyDrive/코드잇_파트2_3팀_프로젝트/project1-data/train_annotations/K-003351-021325-031863_json/K-031863/K-003351-021325-031863_0_2_0_2_70_000_200.json
/content/gdrive/MyDrive/코드잇_파트2_3팀_프로젝트/project1-data/train_annotations/K-003351-021325-031863_json/K-031863/K-003351-021325-031863_0_2_0_2_75_000_200.json
/content/gdrive/MyDrive/코드잇_파트2_3팀_프로젝트/project1-data/train_annotations/K-003351-021325-031863_json/K-003351/K-003351-021325-031863_0_2_0_2_70_000_200.json
/content/gdrive/MyDrive/코드잇_파트2_3팀_프로젝트/project1-data/train_annotations/K-003351-021325-031863_json/K-003351/K-003351-021325-031863_0_2_0_2_75_000_200.json
/content/gdrive/MyDrive/코드잇_파트2_3팀_프로젝트/project1-data/train_annotations/K-003351-021325-031863_json/K-021325/K-003351-021325-031863_0_2_0_2_70_000_200.json


In [18]:
target_classes = {}

base_json_files = TRAIN_ANNOTATION_DIR.rglob("*.json")

for json_path in base_json_files:
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for category in data.get("categories", []):
            target_classes[str(category["id"])] = category.get("name", "")

    except Exception as e:
        print(f"읽기 실패: {json_path.name} -> {e}")

print("기존 프로젝트 클래스 수:", len(target_classes))

기존 프로젝트 클래스 수: 56


In [19]:
target_df = pd.DataFrame(
    [
        {
            "category_id": category_id,
            "category_name": category_name
        }
        for category_id, category_name in target_classes.items()
    ]
)

target_df.head()

,category_id,category_name
0,31863,아질렉트정(라사길린메실산염)
1,3351,일양하이트린정 2mg
2,21325,아토르바정 10mg
3,20238,플라빅스정 75mg
4,20014,마도파정


### 7. 추가 AI Hub 데이터 내 기존 클래스 검색

각 TL ZIP 파일 내부의 JSON을 압축 해제하지 않고 순차적으로 읽어,
현재 프로젝트의 56개 클래스와 동일한 약품 ID가 존재하는지 확인합니다.

전체 JSON 내용을 저장하지 않고 일치하는 데이터만 수집하여
대용량 라벨링 데이터 탐색 시 메모리 사용을 최소화합니다.

In [26]:
matched_records = []
scan_summary = []

for zip_path in zip_files:

    scanned = 0
    matched = 0
    errors = 0

    print(f"\n[검색 시작] {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:

        json_names = [
            name for name in z.namelist()
            if name.lower().endswith(".json")
        ]

        print(f"JSON 수: {len(json_names):,}")

        for json_name in json_names:

            try:
                with z.open(json_name) as f:
                    data = json.load(f)

                scanned += 1

                images = data.get("images", [])

                for img in images:

                    dl_idx = img.get("dl_idx")

                    if dl_idx is None:
                        continue

                    dl_idx = str(dl_idx)

                    if dl_idx in target_classes:

                        matched += 1

                        matched_records.append({
                            "zip_file": zip_path.name,
                            "json_file": json_name,
                            "dl_idx": dl_idx,
                            "dl_name": img.get("dl_name"),
                            "file_name": img.get("file_name"),
                        })

            except Exception:
                errors += 1

    scan_summary.append({
        "zip_file": zip_path.name,
        "json_count": scanned,
        "matched_count": matched,
        "error_count": errors,
    })

    print(
        f"완료 → 검색 {scanned:,}건 / "
        f"일치 {matched:,}건 / "
        f"오류 {errors:,}건"
    )


[검색 시작] TL_10_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 1,296건 / 오류 0건

[검색 시작] TL_11_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_12_단일.zip
JSON 수: 64,584
완료 → 검색 64,584건 / 일치 1,296건 / 오류 0건

[검색 시작] TL_13_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_14_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 1,296건 / 오류 0건

[검색 시작] TL_15_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_16_단일.zip
JSON 수: 64,584
완료 → 검색 64,584건 / 일치 1,296건 / 오류 0건

[검색 시작] TL_17_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_18_단일.zip
JSON 수: 64,698
완료 → 검색 64,698건 / 일치 0건 / 오류 0건

[검색 시작] TL_19_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_1_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_20_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_21_단일.zip
JSON 수: 29,592
완료 → 검색 29,592건 / 일치 0건 / 오류 0건

[검색 시작] TL_22_단일.zip
JSON 수: 25

In [27]:
# 추가 AI Hub 샘플의 약품 ID 확인
if "images" in sample_data:
    for img in sample_data["images"][:5]:
        print(
            "dl_idx:", img.get("dl_idx"),
            "| dl_name:", img.get("dl_name")
        )

dl_idx: 249 | dl_name: 마그밀정(수산화마그네슘)


In [28]:
print("기존 56개 ID 샘플:")
print(list(target_classes.items())[:10])

기존 56개 ID 샘플:
[('31863', '아질렉트정(라사길린메실산염)'), ('3351', '일양하이트린정 2mg'), ('21325', '아토르바정 10mg'), ('20238', '플라빅스정 75mg'), ('20014', '마도파정'), ('38162', '로수바미브정 10/20mg'), ('41768', '카발린캡슐 25mg'), ('16688', '오마코연질캡슐(오메가-3-산에틸에스테르90)'), ('3483', '기넥신에프정(은행엽엑스)(수출용)'), ('35206', '아토젯정 10/40mg')]


#### 7-1. 검색 결과 검증


추가 AI Hub 라벨링 데이터에서 확인된 약품 ID와
기존 프로젝트의 56개 클래스 ID를 비교하여 실제 교집합을 확인합니다.

In [29]:
matched_records = []
scan_summary = []

additional_classes = {}

for zip_path in zip_files:

    scanned = 0
    matched = 0
    errors = 0

    print(f"\n[검색 시작] {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:

        json_names = [
            name for name in z.namelist()
            if name.lower().endswith(".json")
        ]

        print(f"JSON 수: {len(json_names):,}")

        for json_name in json_names:

            try:
                with z.open(json_name) as f:
                    data = json.load(f)

                scanned += 1

                for img in data.get("images", []):

                    dl_idx = img.get("dl_idx")

                    if dl_idx is None:
                        continue

                    dl_idx = str(dl_idx)
                    dl_name = img.get("dl_name", "")

                    # 추가 데이터에 존재하는 클래스 기록
                    additional_classes[dl_idx] = dl_name

                    # 기존 56개와 비교
                    if dl_idx in target_classes:

                        matched += 1

                        matched_records.append({
                            "zip_file": zip_path.name,
                            "json_file": json_name,
                            "dl_idx": dl_idx,
                            "dl_name": dl_name,
                            "file_name": img.get("file_name"),
                        })

            except Exception:
                errors += 1

    scan_summary.append({
        "zip_file": zip_path.name,
        "json_count": scanned,
        "matched_count": matched,
        "error_count": errors,
    })

    print(
        f"완료 → 검색 {scanned:,}건 / "
        f"일치 {matched:,}건 / "
        f"오류 {errors:,}건"
    )


[검색 시작] TL_10_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 1,296건 / 오류 0건

[검색 시작] TL_11_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_12_단일.zip
JSON 수: 64,584
완료 → 검색 64,584건 / 일치 1,296건 / 오류 0건

[검색 시작] TL_13_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_14_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 1,296건 / 오류 0건

[검색 시작] TL_15_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_16_단일.zip
JSON 수: 64,584
완료 → 검색 64,584건 / 일치 1,296건 / 오류 0건

[검색 시작] TL_17_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_18_단일.zip
JSON 수: 64,698
완료 → 검색 64,698건 / 일치 0건 / 오류 0건

[검색 시작] TL_19_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_1_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_20_단일.zip
JSON 수: 64,800
완료 → 검색 64,800건 / 일치 0건 / 오류 0건

[검색 시작] TL_21_단일.zip
JSON 수: 29,592
완료 → 검색 29,592건 / 일치 0건 / 오류 0건

[검색 시작] TL_22_단일.zip
JSON 수: 25

In [30]:
target_ids = set(target_classes.keys())
additional_ids = set(additional_classes.keys())

common_ids = target_ids & additional_ids

print("기존 클래스:", len(target_ids))
print("추가 데이터 클래스:", len(additional_ids))
print("일치 클래스:", len(common_ids))

기존 클래스: 56
추가 데이터 클래스: 4023
일치 클래스: 16


### 8. 일치 Annotation 저장 폴더 생성

추가 AI Hub 데이터에서 기존 56개 클래스와 일치하는 Annotation을
별도의 폴더에 저장하여 이후 추가 데이터 확보에 활용합니다.

약품 클래스별로 폴더를 생성하여 일치한 JSON 파일을 구분하여 저장합니다.

In [31]:
MATCHED_OUTPUT_DIR = (
    MY_DRIVE / "pill_matched_annotations"
)

MATCHED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("저장 경로:", MATCHED_OUTPUT_DIR)

저장 경로: /content/gdrive/MyDrive/pill_matched_annotations


### 9. 일치 Annotation 파일 저장

기존 56개 클래스와 일치하는 Annotation JSON 파일을
ZIP 내부에서 추출하여 별도의 검수용 폴더에 저장합니다.

약품 ID별로 하위 폴더를 생성하여 클래스별로 구분해 저장합니다.

In [33]:
# 검색 결과 DataFrame 생성
matched_df = pd.DataFrame(matched_records)

print("일치 Annotation 수:", len(matched_df))

if not matched_df.empty:
    display(matched_df.head())
else:
    print("현재 일치하는 Annotation이 없습니다.")

일치 Annotation 수: 13608


,zip_file,json_file,dl_idx,dl_name,file_name
0,TL_10_단일.zip,K-019553_json/K-019553_0_0_0_0_60_000_200.json,19552,콘택골드캡슐 10mg/PTP,K-019553_0_0_0_0_60_000_200.png
1,TL_10_단일.zip,K-019553_json/K-019553_0_0_0_0_60_020_200.json,19552,콘택골드캡슐 10mg/PTP,K-019553_0_0_0_0_60_020_200.png
2,TL_10_단일.zip,K-019553_json/K-019553_0_0_0_0_60_040_200.json,19552,콘택골드캡슐 10mg/PTP,K-019553_0_0_0_0_60_040_200.png
3,TL_10_단일.zip,K-019553_json/K-019553_0_0_0_0_60_060_200.json,19552,콘택골드캡슐 10mg/PTP,K-019553_0_0_0_0_60_060_200.png
4,TL_10_단일.zip,K-019553_json/K-019553_0_0_0_0_60_080_200.json,19552,콘택골드캡슐 10mg/PTP,K-019553_0_0_0_0_60_080_200.png


In [ ]:
from tqdm.auto import tqdm
import zipfile
from pathlib import Path

# 저장할 ZIP 경로
MATCHED_ZIP_PATH = (
    MATCHED_OUTPUT_DIR / "matched_annotations.zip"
)

saved_count = 0
error_count = 0

with zipfile.ZipFile(
    MATCHED_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as output_zip:

    for _, row in tqdm(
        matched_df.iterrows(),
        total=len(matched_df),
        desc="일치 Annotation 압축 중"
    ):
        try:
            source_zip_path = (
                ADDITIONAL_DATA_DIR / row["zip_file"]
            )

            json_name = row["json_file"]
            dl_idx = str(row["dl_idx"])

            # 원본 TL ZIP에서 JSON 읽기
            with zipfile.ZipFile(source_zip_path, "r") as source_zip:
                raw_data = source_zip.read(json_name)

            # ZIP 내부 저장 경로
            # 예: 19552/TL_10_단일__K-019553....json
            output_name = (
                f"{dl_idx}/"
                f"{source_zip_path.stem}__{Path(json_name).name}"
            )

            output_zip.writestr(
                output_name,
                raw_data
            )

            saved_count += 1

        except Exception as e:
            error_count += 1

print("\n압축 저장 완료!")
print("저장된 Annotation:", saved_count)
print("오류:", error_count)
print("ZIP 저장 위치:", MATCHED_ZIP_PATH)

새로 저장된 Annotation: 13608
저장 위치: /content/gdrive/MyDrive/pill_matched_annotations


In [35]:
print("일치 Annotation 수:", len(matched_df))
print("고유 이미지 수:", matched_df["file_name"].nunique())
print("일치 클래스 수:", matched_df["dl_idx"].nunique())

일치 Annotation 수: 13608
고유 이미지 수: 13608
일치 클래스 수: 16


In [36]:
class_summary = (
    matched_df
    .groupby(["dl_idx", "dl_name"])
    .agg(
        annotation_count=("json_file", "count"),
        image_count=("file_name", "nunique")
    )
    .reset_index()
    .sort_values("image_count", ascending=False)
)

display(class_summary)

,dl_idx,dl_name,annotation_count,image_count
0,12247,아빌리파이정 15mg,1296,1296
2,16232,리피토정 40mg,1296,1296
6,19552,콘택골드캡슐 10mg/PTP,1296,1296
5,1900,보령부스파정 10mg,1296,1296
12,30308,트라젠타듀오정 2.5/1000mg,1296,1296
10,25469,아모잘탄정 5/50mg,1296,1296
9,22347,자누비아정 100mg,1296,1296
8,20877,엑스포지정 5/80mg,1296,1296
14,35206,아토젯정 10/80mg,1296,1296
4,18147,리리카캡슐 300mg,324,324
